# addons

> Google Workspace Add-ons deployment and test-installation lifecycle.

In [ ]:
#| default_exp addons

In [ ]:
#| export
from fastcore.utils import *
from fastgws.core import GWSTransport

`WorkspaceAddons` manages HTTP add-on deployments and per-user test installation without depending on Google's discovery endpoint. It uses fastgws' authenticated, refreshing, retrying transport against the documented REST paths; the discovery-backed `GWSApi` remains the default for services whose discovery document is available.

In [ ]:
#| export
class WorkspaceAddons:
    "Workspace Add-ons deployment and test-installation client"
    scope = 'https://www.googleapis.com/auth/cloud-platform'
    base_url = 'https://gsuiteaddons.googleapis.com/v1'

    def __init__(self,
        project:str, # Google Cloud project ID or number
        creds, # Google OAuth credentials with `scope`
        timeout:float=60, # Request timeout in seconds
    ):
        self.project,self.parent = project,f'projects/{project}'
        headers = {'Accept-Encoding':'gzip', 'User-Agent':'fastgws (gzip)', 'X-Goog-User-Project':project}
        self.transport = GWSTransport(timeout=timeout, base_headers=headers, creds=creds)

    def deployment_name(self, deployment_id:str): return f'{self.parent}/deployments/{deployment_id}'

    async def _request(self, method, path, **kwargs):
        return await self.transport.request(method, f'{self.base_url}/{path}', **kwargs)

## Project and deployments

Authorization exposes the service-account identity Google uses for system ID tokens and the OAuth client audience used for user ID tokens. Deployment listing is a read-only way to inspect the project before changing it.

In [ ]:
#| export
@patch
async def authorization(self:WorkspaceAddons):
    "Get the project's add-on callback authorization identities"
    return await self._request('GET', f'{self.parent}/authorization')

@patch
async def list_deployments(self:WorkspaceAddons):
    "List the project's deployments"
    return await self._request('GET', f'{self.parent}/deployments')

`deploy` uses the API's replace operation: the same call creates a missing deployment or replaces an existing one. This gives iterative tooling one idempotent operation instead of a list-then-create-or-update branch.

In [ ]:
#| export
@patch
async def deploy(self:WorkspaceAddons,
    deployment_id:str, # Stable deployment ID
    manifest:dict, # Deployment resource, without requiring its `name`
):
    "Create or replace a deployment"
    name = self.deployment_name(deployment_id)
    return await self._request('PUT', name, json={**manifest, 'name':name})

## Test installation

Test installation belongs to the authenticated user, so use a separate `WorkspaceAddons` instance for each tester's credentials. `ensure_installed` is the idempotent initial setup operation. After replacing a live development deployment, `reinstall` deliberately uninstalls and installs it again so host applications pick up the new callback configuration.

In [ ]:
#| export
@patch
async def install_status(self:WorkspaceAddons, deployment_id:str):
    "Get this user's test-installation status"
    return await self._request('GET', f'{self.deployment_name(deployment_id)}/installStatus')

@patch
async def install(self:WorkspaceAddons, deployment_id:str):
    "Install a deployment for this user to test"
    return await self._request('POST', f'{self.deployment_name(deployment_id)}:install')

@patch
async def uninstall(self:WorkspaceAddons, deployment_id:str):
    "Uninstall a test deployment from this user"
    return await self._request('POST', f'{self.deployment_name(deployment_id)}:uninstall')

@patch
async def ensure_installed(self:WorkspaceAddons, deployment_id:str):
    "Install a deployment when needed and return its current status"
    status = await self.install_status(deployment_id)
    if not status.installed: await self.install(deployment_id)
    return status if status.installed else await self.install_status(deployment_id)

@patch
async def reinstall(self:WorkspaceAddons, deployment_id:str):
    "Reinstall a deployment and return its current status"
    status = await self.install_status(deployment_id)
    if status.installed: await self.uninstall(deployment_id)
    await self.install(deployment_id)
    return await self.install_status(deployment_id)

A complete first-time setup reads the callback identities, replaces one stable deployment, and ensures the active tester has it installed. Subsequent callback changes use `reinstall` after `deploy`:

In [ ]:
#| eval: false
creds = await oauth_creds(account='developer@example.com', internal=True,
    scopes=[WorkspaceAddons.scope])
addons = WorkspaceAddons('my-project', creds)
auth = await addons.authorization()
deployment = await addons.deploy('dev', manifest)
status = await addons.ensure_installed('dev')

updated = await addons.deploy('dev', updated_manifest)
refreshed = await addons.reinstall('dev')
(auth.serviceAccountEmail, updated.name, refreshed.installed)